In [1]:
# ── CELL 3 ── Mount Google Drive and load data ─────────────────────────────────

# %%
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install "transformers<4.46.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 99.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 55.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 109.9 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.8.0
    Uninstalling huggingface_hub-1.8.0:
      Successfully uninstalled huggingface_hub-1.8.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [3]:
!pip install "trl<=0.9.6"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.8/245.8 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 80.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 21.4 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
tobler 0.13.0 requires numpy>=2.0,

In [4]:
# ==============================================================================
# QLoRA Fine-Tuning: Ministral-8B-Instruct for Defense Mechanism Classification
# Target: PsyDefDetect @ BioNLP 2026
# Hardware: Google Colab T4 (16 GB VRAM)
# ==============================================================================

# ── CELL 1 ── Install dependencies ────────────────────────────────────────────
# Run once; restart runtime after installation.

# %%
!pip install -q \
    datasets \
    peft \
    bitsandbytes \
    accelerate \
    huggingface_hub \
    sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.5 MB/s eta 0:00:00


In [1]:

# ── CELL 2 ── Imports ──────────────────────────────────────────────────────────

# %%
import json
import os
import torch
from pathlib import Path

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from peft import LoraConfig, TaskType
from trl import SFTTrainer, SFTConfig, DataCollatorForCompletionOnlyLM

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    print(f"VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")



PyTorch version : 2.10.0+cu128
CUDA available  : True
GPU             : Tesla T4
VRAM            : 15.6 GB


In [2]:


# ── Edit this path to wherever you placed the file in your Drive ──
DATA_PATH = "/content/drive/MyDrive/BioNLP_Central/balanced_train_data_with_trace.json"
OUTPUT_DIR = "/content/drive/MyDrive/BioNLP_Central/output/"

with open(DATA_PATH, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

print(f"Loaded {len(raw_data)} training examples")
print("Sample keys:", list(raw_data[0].keys()))

Loaded 758 training examples
Sample keys: ['id', 'dialogue_id', 'dialogue', 'current_text', 'label', 'predicted_label', 'predicted_defense_level', 'clinical_reasoning', 'model_thinking_dump']


In [3]:


# ── CELL 4 ── Prompt templates ─────────────────────────────────────────────────

# %%
SYSTEM_PROMPT = """\
You are an expert clinical psychologist analyzing conversational data. Your task is to evaluate the psychological defense mechanism used in a target utterance based on the Defense Mechanisms Rating Scales (DMRS).

LABEL REFERENCE (Defense Mechanism Rating Scale Tiers):
  0 = No Defense / Neutral Utterance  — simple, undefended statement; no psychological distortion - Functional utterances that maintain conversational flow without engaging conflict.
  1 = Action Defense Level            — Acting Out / Help-Rejecting Complaining / Passive Aggression - Distress is released by acting on the environment instead of reflecting.
  2 = Major Image-distorting Defense  — Splitting / Projective Identification - Reduces anxiety via all-good/all-bad distortions of self or other.
  3 = Disavowal Defense Level         — Denial / Projection / Rationalization / Autistic Fantasy - Rejects threatening reality by denying, excusing, blaming, or fantasizing.
  4 = Minor Image-distorting Defense  — Devaluation / Idealization / Omnipotence - Softer distortions temporarily inflate or deflate self-esteem.
  5 = Neurotic Defense Level          — Displacement / Dissociation / Reaction Formation / Repression - Keeps unacceptable motives out of awareness; feelings surface indirectly.
  6 = Obsessional Defense Level       — Intellectualization / Isolation of Affects / Undoing - Uses excessive logic or symbolic acts to separate feelings from events.
  7 = Highly Adaptive Defense Level   — Affiliation / Altruism / Anticipation / Humor / Self-Assertion / Self-Observation / Sublimation / Suppression - Mature coping that integrates emotion and thought to channel affect constructively.
  8 = Need More Information           — Evidence suggests a defense but is insufficient to confirm any tier - Label used when an utterance is too ambiguous or lacks context.

Follow these analytical steps:
1. Context   : Analyze the preceding dialogue to understand what triggered this utterance.
2. Function  : Identify the psychological goal the speaker is trying to achieve or avoid.
3. Grounding : Match the behavior to specific DMRS criteria.
4. Hierarchy : Verify that exclusionary criteria for higher/lower levels are met.

First provide your clinical reasoning trace, then output the defense level (0–8).\
"""


def format_dialogue(dialogue: list, current_text: str) -> str:
    """
    Renders the dialogue as readable turns.
    The target utterance is included in its natural position so the model
    sees it in context, then we flag it explicitly below.
    """
    lines = []
    for turn in dialogue:
        speaker = "Seeker" if turn["speaker"] == "seeker" else "Supporter"
        lines.append(f"{speaker}: {turn['text']}")
    return "\n".join(lines)


def build_user_message(example: dict) -> str:
    dialogue_str = format_dialogue(example["dialogue"], example["current_text"])
    return (
        f"## Dialogue Context\n\n"
        f"{dialogue_str}\n\n"
        f"## Target Utterance To Classify: {example['current_text']}"
    )

def build_assistant_response(example: dict) -> str:
    """
    Constructs the chain-of-thought response from the clinical_reasoning dict.
    The final line is the parseable defense level integer.
    """
    cr = example["clinical_reasoning"]
    return (
        f"\nContext Trigger: {cr['context_trigger']}\n\n"
        f"Psychological Goal: {cr['psychological_goal']}\n\n"
        f"Handbook Alignment: {cr['handbook_alignment']}\n\n"
        f"Differential Diagnosis: {cr['differential_diagnosis']}\n\n"
        f"Defense Level: {example['label']}"
    )

# Quick sanity check on one example
sample = raw_data[4]
print("=== USER MESSAGE ===")
print(build_user_message(sample), "...\n")
print("=== ASSISTANT RESPONSE ===")
print(build_assistant_response(sample))

=== USER MESSAGE ===
## Dialogue Context

Supporter: Hi, what can I do to help you?
Seeker: I am having such a hard time because Mom does not deserve this. I wanted her to grow old in love and happy. Now, it's over. I am just so very sad for her.hello hi
Supporter: hi Thanks for your sharing and sorry to hear about that
Seeker: Ty. Death is so hard. How would you deal?
Supporter: I felt the same way as you when your closed family members pass away I would share my feeling with someone that I felt closed or some family members or your spouse that you can share your feeling. Do you try to talk to anyone about your feeling? Or someone that close to you that you can share your feeling with?
Seeker: My husband is here. He is very afraid of the virus because of his COPD. He i afraid to talk about the virus. I just moved to this town where there are not so many people because of thi.
Supporter: Maybe try to talk about something that he is very interesting and help him away of focusing on COVI

In [ ]:
# ── CELL 5 ── Build HuggingFace Dataset ───────────────────────────────────────

# %%
from datasets import ClassLabel
import torch

MODEL_ID = "mistralai/Ministral-3-8B-Reasoning-2512"

# Load tokenizer first so we can apply the chat template during formatting
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"   # required for SFTTrainer

# Note: The Reasoning model might have special reasoning tags. 
# We just use the standard template, assuming text fine-tuning.

def format_for_training(example: dict) -> dict:
    # 1. Manually combine the system and user prompts
    combined_user_content = f"{SYSTEM_PROMPT}\n\n{build_user_message(example)}"

    # 2. Pass ONLY user and assistant roles to the tokenizer
    messages = [
        {"role": "user", "content": combined_user_content},
        {"role": "assistant", "content": build_assistant_response(example)},
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )

    return {"text": text, "label": example["label"]}

# Create the initial dataset with both 'text' and 'label' columns
formatted_examples = [format_for_training(ex) for ex in raw_data]
dataset = Dataset.from_list(formatted_examples)

# --- THE FIX: Cast the integer label to a Hugging Face ClassLabel (0-8 = 9 classes) ---
dataset = dataset.cast_column("label", ClassLabel(num_classes=9))

# Now the stratified split will work!
dataset = dataset.train_test_split(
    test_size=0.1,
    seed=42,
    stratify_by_column="label"
)

# Remove the label column so SFTTrainer doesn't get confused
train_dataset = dataset["train"].remove_columns("label")
eval_dataset  = dataset["test"].remove_columns("label")

print(f"Train size : {len(train_dataset)}")
print(f"Eval  size : {len(eval_dataset)}")
print("\nSample formatted text:\n")
print(train_dataset[0]["text"])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.llama.tokenization_llama_fast.LlamaTokenizerFast'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565 - if you loaded a llama tokenizer from a GGUF file you can ignore this message.


Casting the dataset:   0%|          | 0/758 [00:00<?, ? examples/s]

Train size : 682
Eval  size : 76

Sample formatted text:

<s>[INST]You are an expert clinical psychologist analyzing conversational data. Your task is to evaluate the psychological defense mechanism used in a target utterance based on the Defense Mechanisms Rating Scales (DMRS).

LABEL REFERENCE (Defense Mechanism Rating Scale Tiers):
  0 = No Defense / Neutral Utterance  — simple, undefended statement; no psychological distortion - Functional utterances that maintain conversational flow without engaging conflict.
  1 = Action Defense Level            — Acting Out / Help-Rejecting Complaining / Passive Aggression - Distress is released by acting on the environment instead of reflecting.
  2 = Major Image-distorting Defense  — Splitting / Projective Identification - Reduces anxiety via all-good/all-bad distortions of self or other.
  3 = Disavowal Defense Level         — Denial / Projection / Rationalization / Autistic Fantasy - Rejects threatening reality by denying, excusing, blaming,

In [ ]:
print(tokenizer.chat_template)

{%- if messages[0]["role"] == "system" %}
    {%- set system_message = messages[0]["content"] %}
    {%- set loop_messages = messages[1:] %}
{%- else %}
    {%- set loop_messages = messages %}
{%- endif %}
{%- if not tools is defined %}
    {%- set tools = none %}
{%- endif %}
{%- set user_messages = loop_messages | selectattr("role", "equalto", "user") | list %}

{#- This block checks for alternating user/assistant messages, skipping tool calling messages #}
{%- set ns = namespace() %}
{%- set ns.index = 0 %}
{%- for message in loop_messages %}
    {%- if not (message.role == "tool" or message.role == "tool_results" or (message.tool_calls is defined and message.tool_calls is not none)) %}
        {%- if (message["role"] == "user") != (ns.index % 2 == 0) %}
            {{- raise_exception("After the optional system message, conversation roles must alternate user/assistant/user/assistant/...") }}
        {%- endif %}
        {%- set ns.index = ns.index + 1 %}
    {%- endif %}
{%- endfor

In [ ]:
# ── CELL 7 ── Load model with 4-bit QLoRA quantization ────────────────────────

# %%
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",          # NormalFloat4 — best for LLM weights
    bnb_4bit_compute_dtype=torch.float16, # Fallback to fp16 for T4 compatibility 
    bnb_4bit_use_double_quant=True,      # nested quantization saves ~0.4 GB
)

# For multimodal reasoning models, we load the text causal LM.
# However, if it's explicitly a VLM, `AutoModelForCausalLM` might fail or load 
# vision parameters. We manually pop/delete components we don't need after loading.
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.float16, # T4 works best with fp16
)
model.config.use_cache = False          # required with gradient checkpointing

# Strip away vision components to save VRAM if present (e.g., Pixtral/Ministral vision tower)
if hasattr(model, "vision_tower"):
    del model.vision_tower
if hasattr(model, "vision_model"):
    del model.vision_model
if hasattr(model, "embed_tokens") and hasattr(model.embed_tokens, "vision_embed"):
    del model.embed_tokens.vision_embed

# Force clean up
import gc
gc.collect()
torch.cuda.empty_cache()

print("Model loaded. Parameter breakdown:")
total   = sum(p.numel() for p in model.parameters())
print(f"  Total parameters : {total/1e9:.2f}B")

{'text': Value('string')}


In [ ]:
# ── CELL 8 ── LoRA configuration ──────────────────────────────────────────────

# %%
from peft import LoraConfig, TaskType

# Target all attention + MLP projection layers for maximum expressivity
# r=16, alpha=32 is a safe default; increase r if you have more data
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",   # attention
        "gate_proj", "up_proj", "down_proj",        # MLP (SwiGLU)
    ],
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters : {trainable/1e6:.2f}M "
      f"({100*trainable/total:.2f}% of total)")

# ── CELL 9 ── Completion-only data collator ───────────────────────────────────
# This is critical: loss is computed ONLY on the assistant response,
# not on the system prompt or user message.

# %%
from trl import DataCollatorForCompletionOnlyLM

# Instead of hardcoding '[/INST]', we can extract the assistant boundary
# dynamically from the tokenizer's chat template.
dummy_msgs = [{"role": "user", "content": "test"}]
dummy_str = tokenizer.apply_chat_template(dummy_msgs, tokenize=False, add_generation_prompt=True)

# Try splitting on "test" to see what token sequence signals the assistant response
# For mistral it might be "[/INST]" or "<|assistant|>".
response_template = dummy_str.split("test")[-1].strip()
if not response_template:
    response_template = "[/INST]" # Fallback

print(f"Using dynamically detected response template: {repr(response_template)}")

# You can pass the encoded string ids directly to the collator if the string match is unreliable
response_ids = tokenizer.encode(response_template, add_special_tokens=False)

collator = DataCollatorForCompletionOnlyLM(
    response_template=response_ids, # Preferred over string in newer TRL
    tokenizer=tokenizer,
)

Min    : 673
Median : 1153
95th % : 1577
Max    : 2034

Using MAX_SEQ_LENGTH = 1641


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.07G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Model loaded. Parameter breakdown:
  Total parameters : 4.55B
Trainable parameters : 1074.04M (23.62% of total)


In [ ]:
# ── CELL 10 ── Training configuration ─────────────────────────────────────────

# %%
from trl import SFTConfig, SFTTrainer

# For T4 (16 GB):
#   - batch_size=1 is required to avoid OOM with 8B model + activations
#   - gradient_accumulation=8 gives effective batch size of 8
#   - fp16=True (T4 does not support bfloat16)
#   - gradient_checkpointing trades compute for memory

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,

    # ── Data ──────────────────────────────────────────────────────
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_text_field="text",
    packing=False,

    # ── Batch / memory ────────────────────────────────────────────
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,      # effective batch = 8
    gradient_checkpointing=True,

    # ── Optimizer ─────────────────────────────────────────────────
    num_train_epochs=3,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    weight_decay=0.01,
    optim="paged_adamw_8bit",           # 8-bit AdamW saves optimizer VRAM

    # ── Precision ─────────────────────────────────────────────────
    fp16=True,                          # T4 supports fp16 but NOT bf16
    bf16=False,

    # ── Logging / saving ──────────────────────────────────────────
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none",                   # set to "wandb" if you use W&B

    # ── Reproducibility ───────────────────────────────────────────
    seed=42,
)

# ── CELL 11 ── Trainer ────────────────────────────────────────────────────────

# %%
trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=collator,
    peft_config=lora_config,
)

Map:   0%|          | 0/682 [00:00<?, ? examples/s]

Map:   0%|          | 0/76 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:408: UserWarning: You passed a tokenizer with `padding_side` not equal to `right` to the SFTTrainer. This might lead to some unexpected behaviour due to overflow issues when training a model in half-precision. You might consider adding `tokenizer.padding_side = 'right'` to your code.
  warnings.warn(


In [ ]:
# ── CELL 12 ── Train ──────────────────────────────────────────────────────────

# %%
print("Starting training …")
trainer.train() #resume_from_checkpoint=True

# Save the LoRA adapter (small file, ~50–100 MB) to Drive
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Adapter saved to {OUTPUT_DIR}")

Starting training …


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss


In [ ]:


# ── CELL 13 ── Inference helper ───────────────────────────────────────────────

# %%
import re

def predict_defense_level(
    dialogue: list,
    current_text: str,
    model,
    tokenizer,
    max_new_tokens: int = 512,
) -> dict:
    """
    Runs inference on a single example.
    Returns a dict with:
        - 'reasoning': the full chain-of-thought text
        - 'level': the predicted integer (0–8), or -1 if parsing fails
    """
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": build_user_message(
                {"dialogue": dialogue, "current_text": current_text}
            ),
        },
    ]
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,   # True at inference: model should continue
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,           # greedy at evaluation time
            temperature=1.0,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Decode only the newly generated tokens (strip the prompt)
    generated = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True,
    ).strip()

    # Parse the final "Defense Level: X" line
    match = re.search(r"Defense Level:\s*([0-8])", generated)
    level = int(match.group(1)) if match else -1

    return {"reasoning": generated, "level": level}


# ── CELL 14 ── Quick test on a held-out example ───────────────────────────────

# %%
# Load adapter on top of a freshly quantized base model (or reuse trainer.model)
test_example = raw_data[0]   # swap with a genuine test-split example

result = predict_defense_level(
    dialogue=test_example["dialogue"],
    current_text=test_example["current_text"],
    model=trainer.model,
    tokenizer=tokenizer,
)

print("=== GENERATED REASONING ===")
print(result["reasoning"])
print(f"\n=== PREDICTED LEVEL : {result['level']} ===")
print(f"=== GOLD LEVEL      : {test_example['label']}  ===")

# ── CELL 15 ── Batch evaluation (accuracy on eval split) ─────────────────────

# %%
correct = 0
total   = 0
errors  = []

# Reload raw eval indices (we need the original dicts, not formatted text)
eval_indices = list(range(int(len(raw_data) * 0.9), len(raw_data)))

for idx in eval_indices:
    ex = raw_data[idx]
    result = predict_defense_level(
        dialogue=ex["dialogue"],
        current_text=ex["current_text"],
        model=trainer.model,
        tokenizer=tokenizer,
    )
    gold = ex["label"]
    pred = result["level"]
    if pred == gold:
        correct += 1
    else:
        errors.append({"id": ex["id"], "gold": gold, "pred": pred})
    total += 1

print(f"Accuracy : {correct}/{total} = {correct/total*100:.1f}%")
print(f"Parse failures (level=-1) : {sum(1 for e in errors if e['pred'] == -1)}")